B"H

# Milestone 4: Pulling Data from the API, Cleaning, and Formatting

- DSC-540
- David Koyrakh
- Professor Catie Williams

## Load and examine the data

For this milestone, I'll be working with data from the US Census Bureau's American Community Survey (ACS) 
5-Year API. This dataset provides detailed social and financial statistics across the United States. 
The ACS is a comprehensive sourc for demographic information, collecting data on topics like household size, income, marital status, and other socioeconomic indicators.

In this milestone, I'll be focusing on cleaning and transforming this data to ensure it's properly formatted and ready for analysis. In the final submission, the data will be combined with our USDA and religiosity datasets to create a comprehensive view of demographic patterns across US states.

In [2]:
# Import necessary libraries
import pandas as pd
import requests
import json

### Retrieve the data from the API

First, I need to ensure that the Census API key variable is available in this environment:

In [3]:
# Load environment variables
from dotenv import load_dotenv
import os

# Load .env file
load_dotenv()

# Get Census API key
CENSUS_API_KEY = os.getenv('CENSUS_API_KEY')

if not CENSUS_API_KEY:
    raise ValueError("Census API key not found in environment variables")

Next, I want to retreive all available variables and their names/definitions. This will allow me to select variables relevant to my project:

In [4]:
api_key = CENSUS_API_KEY

# Base URL for Census API - using Data Profiles endpoint
base_url = "https://api.census.gov/data/2023/acs/acs5/profile/variables"

try:
    # Make API request to get variable definitions
    response = requests.get(base_url + ".json")
    response.raise_for_status()
    
    # Convert response to DataFrame
    variables_dict = response.json()['variables']
    
    # Create DataFrame from the dictionary
    var_df = pd.DataFrame.from_dict(variables_dict, orient='index')
    
    # Filter out annotation variables (those ending in 'A' or 'M')
    var_df = var_df[~var_df.index.str.contains('A$|M$')]
    
    # Filter for just DP02, DP03, and DP05 variables
    dp_vars = var_df[var_df.index.str.contains('^DP02|^DP03|^DP05')]
    
    # Sort by variable name
    dp_vars = dp_vars.sort_index()
    
    # Display the first few rows of each DP group
    print("DP02 (Social Characteristics) Sample:")
    print(dp_vars[dp_vars.index.str.startswith('DP02')].head())
    print("\nDP03 (Economic Characteristics) Sample:")
    print(dp_vars[dp_vars.index.str.startswith('DP03')].head())
    print("\nDP05 (Demographic Characteristics) Sample:")
    print(dp_vars[dp_vars.index.str.startswith('DP05')].head())
    
    # Save full variable list to CSV for reference
    dp_vars.to_csv('census_variables.csv')
    print("\nFull variable list saved to 'census_variables.csv'")

except requests.exceptions.RequestException as e:
    print(f"Error making API request: {e}")
    if hasattr(e, 'response') and e.response is not None:
        print("\nResponse content:", e.response.text)
except json.JSONDecodeError as e:
    print(f"Error parsing JSON response: {e}")
    print("\nFull response text:")
    print(response.text)
except Exception as e:
    print(f"An unexpected error occurred: {e}")

DP02 (Social Characteristics) Sample:
                                                           label  \
DP02PR_0001E      Estimate!!HOUSEHOLDS BY TYPE!!Total households   
DP02PR_0001PE      Percent!!HOUSEHOLDS BY TYPE!!Total households   
DP02PR_0002E   Estimate!!HOUSEHOLDS BY TYPE!!Total households...   
DP02PR_0002PE  Percent!!HOUSEHOLDS BY TYPE!!Total households!...   
DP02PR_0003E   Estimate!!HOUSEHOLDS BY TYPE!!Total households...   

                                                      concept predicateType  \
DP02PR_0001E   Selected Social Characteristics in Puerto Rico           int   
DP02PR_0001PE  Selected Social Characteristics in Puerto Rico           int   
DP02PR_0002E   Selected Social Characteristics in Puerto Rico           int   
DP02PR_0002PE  Selected Social Characteristics in Puerto Rico         float   
DP02PR_0003E   Selected Social Characteristics in Puerto Rico           int   

                group  limit predicateOnly hasGeoCollectionSupport  \
DP02PR_0

I am interested in socioeconomic variables relating to family, employment, and income. Therefore, I'm going to request per-state data for the following variables:
- State names (`NAME`)
- Average household size (`DP02_0016E`)
- Average family size (`DP02_0017E`)
- Percent males 15+ currently married (`DP02_0027PE`)
- Percent females 15+ currently married (`DP02_0033PE`)
- Unemployment rate (`DP03_0009PE`)
- Mean household income (`DP03_0063E`)

In [5]:
# Base URL for Census API - using Data Profiles endpoint
base_url = "https://api.census.gov/data/2023/acs/acs5/profile"

try:
    # Construct API URL for actual data
    # Selected variables based on your requirements
    variables = [
        "NAME",  # State names
        # Household and Family
        "DP02_0016E",  # Average household size
        "DP02_0017E",  # Average family size
        
        # Marital Status
        "DP02_0027PE",  # Percent males 15+ currently married
        "DP02_0033PE",  # Percent females 15+ currently married
        
        # Economic Indicators
        "DP03_0009PE",  # Unemployment rate
        "DP03_0063E",  # Mean household income
    ]
    
    params = {
        "get": ",".join(variables),
        "for": "state:*",  # Get data for all states
        "key": api_key
    }
    
    # Make API request
    response = requests.get(base_url, params=params)
        
    # Only proceed if we got a 200 status code
    if response.status_code == 200:
                
        # Convert response to DataFrame
        data = response.json()
        headers = data[0]
        values = data[1:]
        
        df = pd.DataFrame(values, columns=headers)
        
        print("\nFirst few rows:")
        print(df.head())
        
except requests.exceptions.RequestException as e:
    print(f"Error making API request: {e}")
except json.JSONDecodeError as e:
    print(f"Error parsing JSON response: {e}")
    print("\nFull response text:")
    print(response.text)
except Exception as e:
    print(f"An unexpected error occurred: {e}")


First few rows:
         NAME DP02_0016E DP02_0017E DP02_0027PE DP02_0033PE DP03_0009PE  \
0     Alabama       2.50       3.13        49.8        45.3         4.8   
1      Alaska       2.63       3.26        48.5        49.9         5.8   
2     Arizona       2.54       3.12        48.6        46.6         5.2   
3    Arkansas       2.48       3.08        50.7        47.2         5.1   
4  California       2.86       3.43        48.1        45.1         6.4   

  DP03_0063E state  
0      86225    01  
1     114947    02  
2     104138    04  
3      82554    05  
4     136730    06  


## Step 1: Convert column names into more readable names

The current column names are cryptic and non-descriptive. To make this dataset more usable, I would like to convert the column names into more human-readable form.

First, I will list all column names:

In [10]:
df.columns.values

array(['NAME', 'DP02_0016E', 'DP02_0017E', 'DP02_0027PE', 'DP02_0033PE',
       'DP03_0009PE', 'DP03_0063E', 'state'], dtype=object)

In [11]:
# Rename columns for clarity
df = df.rename(columns={
    'state': 'State ID (numeric)',
    'NAME': 'State (long)',
    'DP02_0016E': 'Avg_Household_Size',
    'DP02_0017E': 'Avg_Family_Size',
    'DP02_0027PE': 'Pct_Males_Currently_Married',
    'DP02_0033PE': 'Pct_Females_Currently_Married',
    'DP03_0009PE': 'Unemployment_Rate',
    'DP03_0063E': 'Mean_Household_Income'
})
df.columns.values

array(['State (long)', 'Avg_Household_Size', 'Avg_Family_Size',
       'Pct_Males_Currently_Married', 'Pct_Females_Currently_Married',
       'Unemployment_Rate', 'Mean_Household_Income', 'State ID (numeric)'],
      dtype=object)

And to briefly confirm they are correct:

In [12]:
df.head()

,State (long),Avg_Household_Size,Avg_Family_Size,Pct_Males_Currently_Married,Pct_Females_Currently_Married,Unemployment_Rate,Mean_Household_Income,State ID (numeric)
0,Alabama,2.50,3.13,49.8,45.3,4.8,86225,01
1,Alaska,2.63,3.26,48.5,49.9,5.8,114947,02
2,Arizona,2.54,3.12,48.6,46.6,5.2,104138,04
3,Arkansas,2.48,3.08,50.7,47.2,5.1,82554,05
4,California,2.86,3.43,48.1,45.1,6.4,136730,06


## Step 2: Create column for abbreviated state

All of the datasets in my project are related by state. To make this relation more explicit and usable in future analysis, I must create a normalized state ID column. In my previous datasets, I have been using the two-letter, all-caps state abbreviations for this. Therefore, in this step, I will be creating a new column simply called 'State', with the abbreviated state identifier populated for this dataset:

### First, define a dictionary mapping each state/region (as it features in raw table data) to the state ID.

In [15]:
state_abbr = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR', 'California': 'CA',
    'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE', 'District of Columbia': 'DC', 'Florida': 'FL',
    'Georgia': 'GA', 'Hawaii': 'HI', 'Idaho': 'ID', 'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA',
    'Kansas': 'KS', 'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD', 'Massachusetts': 'MA',
    'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS', 'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE',
    'Nevada': 'NV', 'New Hampshire': 'NH', 'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY',
    'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK', 'Oregon': 'OR',
    'Pennsylvania': 'PA', 'Puerto Rico': 'PR', 'Rhode Island': 'RI', 'South Carolina': 'SC', 'South Dakota': 'SD', 'Tennessee': 'TN',
    'Texas': 'TX', 'Utah': 'UT', 'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV',
    'Wisconsin': 'WI', 'Wyoming': 'WY'
}

### Next, create the new, normalized `State` column:

In [16]:
# Create new State column with abbreviated state names
df['State'] = df['State (long)'].map(state_abbr)

### Display the full new column alongside the original state column, for verification:

In [18]:
df[['State', 'State (long)']]

,State,State (long)
0,AL,Alabama
1,AK,Alaska
2,AZ,Arizona
3,AR,Arkansas
4,CA,California
5,CO,Colorado
6,CT,Connecticut
7,DE,Delaware
8,DC,District of Columbia
9,FL,Florida


## Step 3: Convert columns into corrected types

APIs often return data all as strings, even when the data is numeric. In our case, that is happening as well:

In [21]:
# Show data types for each column
df.dtypes

State (long)                     object
Avg_Household_Size               object
Avg_Family_Size                  object
Pct_Males_Currently_Married      object
Pct_Females_Currently_Married    object
Unemployment_Rate                object
Mean_Household_Income            object
State ID (numeric)               object
State                            object
dtype: object

The 'object' type is Pandas' way of telling us that all our columns are being treated as strings, even though most of them contain numeric data. Therefore, I need to convert the numeric columns to numeric types:

In [24]:
# Convert numeric columns
# It's all of them besides 'State' and 'State (long)'
numeric_columns = ['Avg_Household_Size', 'Avg_Family_Size', 'Pct_Males_Currently_Married',
                  'Pct_Females_Currently_Married', 'Unemployment_Rate', 'Mean_Household_Income', 
                  'State ID (numeric)']

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col])

df.dtypes

State (long)                      object
Avg_Household_Size               float64
Avg_Family_Size                  float64
Pct_Males_Currently_Married      float64
Pct_Females_Currently_Married    float64
Unemployment_Rate                float64
Mean_Household_Income              int64
State ID (numeric)                 int64
State                             object
dtype: object

All numeric columns have been converted successfully. But, there is one last thing: I have to ensure that the "percent" columns are represented as decimals rather than raw percentage values.

In [27]:
print("Percentage columns pre-transformation:")
df[['Pct_Females_Currently_Married', 'Pct_Males_Currently_Married']].head()

Percentage columns pre-transformation:


,Pct_Females_Currently_Married,Pct_Males_Currently_Married
0,45.3,49.8
1,49.9,48.5
2,46.6,48.6
3,47.2,50.7
4,45.1,48.1


In [28]:
# Convert percentage columns into decimals
pct_columns = ['Pct_Females_Currently_Married', 'Pct_Males_Currently_Married', 'Unemployment_Rate']
for col in pct_columns:
    df[col] = df[col] / 100

print("Percentage columns post-transformation:")
df[['Pct_Females_Currently_Married', 'Pct_Males_Currently_Married']].head()

Percentage columns post-transformation:


,Pct_Females_Currently_Married,Pct_Males_Currently_Married
0,0.453,0.498
1,0.499,0.485
2,0.466,0.486
3,0.472,0.507
4,0.451,0.481


One last preview of our data upon completing this step:

In [29]:
df.head()

,State (long),Avg_Household_Size,Avg_Family_Size,Pct_Males_Currently_Married,Pct_Females_Currently_Married,Unemployment_Rate,Mean_Household_Income,State ID (numeric),State
0,Alabama,2.50,3.13,0.498,0.453,0.048,86225,1,AL
1,Alaska,2.63,3.26,0.485,0.499,0.058,114947,2,AK
2,Arizona,2.54,3.12,0.486,0.466,0.052,104138,4,AZ
3,Arkansas,2.48,3.08,0.507,0.472,0.051,82554,5,AR
4,California,2.86,3.43,0.481,0.451,0.064,136730,6,CA


All of my dataset's columns are now of the correct type.

## Step 4: Handle missing data

In this step, I will check for handle missing data. But first, I have to perform a quick check to see if there are any missing values:

In [31]:
# Check for missing values in the dataset
print("Number of missing values in each column:")
print(df.isnull().sum())

print("\nTotal number of missing values in entire dataset:")
print(df.isnull().sum().sum())

Number of missing values in each column:
State (long)                     0
Avg_Household_Size               1
Avg_Family_Size                  1
Pct_Males_Currently_Married      1
Pct_Females_Currently_Married    1
Unemployment_Rate                0
Mean_Household_Income            0
State ID (numeric)               0
State                            0
dtype: int64

Total number of missing values in entire dataset:
4


Indeed, there are a total of 4 missing values in the dataset. Let's see which rows they are in:

In [32]:
# Display rows with missing values
print("Rows containing missing values:")
print(df[df.isnull().any(axis=1)])

Rows containing missing values:
   State (long)  Avg_Household_Size  Avg_Family_Size  \
51  Puerto Rico                 NaN              NaN   

    Pct_Males_Currently_Married  Pct_Females_Currently_Married  \
51                          NaN                            NaN   

    Unemployment_Rate  Mean_Household_Income  State ID (numeric) State  
51              0.121                  40315                  72    PR  


The same row contains all of the missing values, and it is the row for Puerto Rico. Since Puerto Rico is a territory and not a state, it can be excluded from the project dataset, since I am primarily focusing on states alone.

In [33]:
# Drop the row for Puerto Rico
df = df[df['State'] != 'PR']

# Verify no missing values remain
print("\nNumber of missing values after dropping Puerto Rico:")
print(df.isnull().sum())
print("\nTotal missing values:", df.isnull().sum().sum())


Number of missing values after dropping Puerto Rico:
State (long)                     0
Avg_Household_Size               0
Avg_Family_Size                  0
Pct_Males_Currently_Married      0
Pct_Females_Currently_Married    0
Unemployment_Rate                0
Mean_Household_Income            0
State ID (numeric)               0
State                            0
dtype: int64

Total missing values: 0


## Step 5: Drop unneccessary columns

The last step I will perform to clean and prepare this dataset is dropping unnecessary columns. I want to do this to ensure that my dataset is clean and easily readable and usable for future analysis.

First, I want to preview my data again to see if any columns are not neccessary:

In [34]:
df.head()

,State (long),Avg_Household_Size,Avg_Family_Size,Pct_Males_Currently_Married,Pct_Females_Currently_Married,Unemployment_Rate,Mean_Household_Income,State ID (numeric),State
0,Alabama,2.50,3.13,0.498,0.453,0.048,86225,1,AL
1,Alaska,2.63,3.26,0.485,0.499,0.058,114947,2,AK
2,Arizona,2.54,3.12,0.486,0.466,0.052,104138,4,AZ
3,Arkansas,2.48,3.08,0.507,0.472,0.051,82554,5,AR
4,California,2.86,3.43,0.481,0.451,0.064,136730,6,CA


It appears that the only unneccessary columns are `State ID (numeric)` and `State (long)`, since I will be relying on the abbreviated `State` column when relating and analyzing all of the datasets. Therefore, I can remove those 2 columns:

In [35]:
# Drop unnecessary columns
df = df.drop(['State ID (numeric)', 'State (long)'], axis=1)

# Display the first few rows to verify columns were dropped
print("\nDataset after dropping unnecessary columns:")
df.head()


Dataset after dropping unnecessary columns:


,Avg_Household_Size,Avg_Family_Size,Pct_Males_Currently_Married,Pct_Females_Currently_Married,Unemployment_Rate,Mean_Household_Income,State
0,2.50,3.13,0.498,0.453,0.048,86225,AL
1,2.63,3.26,0.485,0.499,0.058,114947,AK
2,2.54,3.12,0.486,0.466,0.052,104138,AZ
3,2.48,3.08,0.507,0.472,0.051,82554,AR
4,2.86,3.43,0.481,0.451,0.064,136730,CA


## Step 6: Create an aggregate US-level row

I decided to an extra, bonus step: create a row for US-aggregate data. This has the added benefit of showing a high-level summary of the United States.

To create this row, I will simply take the median of all other values, and assign 'US' as the value for `State`:

In [36]:
# Calculate medians for numeric columns
us_row = df.select_dtypes(include=['float64', 'int64']).median()

# Create a new row for US aggregate data
us_row = pd.DataFrame([us_row])
us_row['State'] = 'US'

# Append the US row to the dataframe
df = pd.concat([df, us_row], ignore_index=True)

# Display the last few rows to verify US row was added
print("\nDataset after adding US aggregate row:")
df.tail()



Dataset after adding US aggregate row:


,Avg_Household_Size,Avg_Family_Size,Pct_Males_Currently_Married,Pct_Females_Currently_Married,Unemployment_Rate,Mean_Household_Income,State
47,2.51,3.07,0.513,0.498,0.050,129559.0,WA
48,2.40,3.01,0.491,0.478,0.057,78799.0,WV
49,2.35,2.97,0.505,0.489,0.033,99059.0,WI
50,2.38,2.99,0.530,0.535,0.037,97459.0,WY
51,2.47,3.07,0.502,0.472,0.049,102130.0,US


## Conclusion

The transformations in this notebook included:

1. Converting cryptic Census API column names into human-readable labels
2. Creating a normalized state identifier column using two-letter abbreviations
3. Converting data types from strings to appropriate numeric types and standardizing percentage representations
4. Handling missing data by removing Puerto Rico (territory, not state)
5. Dropping redundant columns to improve dataset usability
6. Creating an aggregate US-level summary row using median values

The dataset comes from the US Census Bureau's American Community Survey (ACS) 5-Year API. The data is public domain and freely available. The data was obtained legally through the Census Bureau's API using proper authentication.

The transformations performed were focused on improving data usability while maintaining data integrity. By removing Puerto Rico, the data did lose some territorial data, but this is justified since I am focusing primarily on state-level national data. I assumed that the Census Bureau's data validation processes ensured data quality, though it would have been ideal to check for anomolies, serious outliers, and unexpected patterns.

To mitigate any risks from our transformations, I maintained clear documentation of all changes made to the original data. The addition of the US-level summary row is clearly marked as derived data, and all percentage conversions preserved the original numerical relationships. Any future use of this dataset should include clear documentation of modifications, especially regarding ranking changes and aggregation methods, to avoid misleading conclusions.